In [ ]:
import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_v3.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)
ragas_df

In [5]:
from ragas import evaluate
from ragas.metrics import (
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
import openai
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.embeddings import GoogleEmbeddings
from google import genai
from dotenv import load_dotenv
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

load_dotenv()

import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_2026-05-22.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# 用 AsyncOpenAI
async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory(
    "gpt-4o-mini",
    provider="openai",
    client=async_client,
    max_tokens=4096,
    max_retries=6,
    timeout=120,  
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

answer_relevancy = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
請用繁體中文產生問題。"""

scores = evaluate(
    dataset=rag_results,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy,
        # AnswerCorrectness(llm=evaluator_llm),
    ],
)


print(scores)

C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_15160\1994392570.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_1516


statement: 蘇彥碩在使用Git時遇到的最大問題是，有人不會使用Git。
verdict: 1
reason: The context explicitly mentions that one of the major problems is that some people do not know how to use Git, which directly relates to 蘇彥碩's experience.


Evaluating:  10%|▉         | 19/200 [00:10<01:13,  2.45it/s]


statement: 在選擇SDGs主題時，需遵循以下原則。
verdict: 1
reason: The context outlines principles for selecting topics, including the SDGs, which supports this statement.

statement: 題目須排除過去三年的題目。
verdict: 1
reason: The context explicitly states that topics must exclude those from the past three years.

statement: 題目選擇應依據TronClass公告為準。
verdict: 1
reason: The context mentions that topic selection should be based on TronClass announcements.

statement: 應盡量選擇可以找到實際使用者進行訪談的題目。
verdict: 1
reason: The context advises to choose topics that allow for interviews with actual users.

statement: 根據SDGs，應挑選一個有興趣的SDG。
verdict: 1
reason: The context instructs to select an interesting SDG based on the SDGs.

statement: 從挑選的SDG中尋找主題。
verdict: 1
reason: The context indicates that one should find a topic from the selected SDG.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  14%|█▍        | 28/200 [00:14<00:57,  2.97it/s]

...


statement: 根據朱浩偉的經驗，朱浩偉在專題製作中建議要安排一個計畫。
verdict: 1
reason: 朱浩偉的經驗中提到要安排計畫以確保小組進度，因此這個陳述可以直接推斷。

statement: 安排計畫可以確保小組進度能夠跟著計劃進行。
verdict: 1
reason: 根據朱浩偉的經驗，安排計畫的確是為了確保小組進度，因此這個陳述是正確的。

statement: 朱浩偉建議把握每個可以寫小專案的機會。
verdict: 1
reason: 朱浩偉的經驗中提到要把握每一個可以寫小專案的機會，因此這個陳述是正確的。

statement: 釐清系統開發對使用者的價值會讓專題製作更順利。
verdict: 1
reason: 朱浩偉提到釐清系統開發對使用者的價值會使專題製作更順利，因此這個陳述是正確的。

statement: 至於時間投入方面，初期每週可能需要10-15小時。
verdict: 1
reason: 朱浩偉提到初期每週可能需要10-15小時的時間投入，因此這個陳述是正確的。

statement: 初期的時間主要用於研究和規劃。
verdict: 1
reason: 根據朱浩偉的經驗，初期的時間確實主要用於研究和規劃，因此這個陳述是正確的。

statement: 在進入開發和整合階段，接近死線或與產學合作方有審查時，每週投入的時間會增加到20-30小時甚至更多。
verdict: 1
reason: 朱浩偉提到在開發和整合階段，時間投入會增加到20-30小時甚至更多，因此這個陳述是正確的。


Evaluating: 100%|█████████▉| 199/200 [01:54<00:01,  1.16s/it]


statement: 林芯緹和蘇彥碩建議在共同開發時要事先溝通好程式的命名規則。
verdict: 1
reason: 根據林芯緹、蘇彥碩的經驗，確實提到要溝通好程式之命名規則，因此這個陳述可以直接推斷。

statement: 程式的命名規則最好以功能來命名。
verdict: 1
reason: 學姐建議以功能命名，這與陳述一致，因此可以直接推斷。

statement: 這樣可以讓所有人以及未來的自己都清楚程式碼的意義。
verdict: 1
reason: 文中提到可以將自己的邏輯想法寫在註解，讓大家和未來的自己都清楚程式碼的意義，因此這個陳述可以直接推斷。

statement: 另外，建議在開發過程中定期舉行「debug 大會」來追蹤組員的進度。
verdict: 1
reason: 文中提到偶爾開個debug 大會，這與陳述一致，因此可以直接推斷。

statement: 在「debug 大會」中，應該適時關心每位組員的狀況。
verdict: 1
reason: 文中提到並適時關心每一個組員進度，這與陳述一致，因此可以直接推斷。

statement: 這樣可以確保大家都在同一頁。
verdict: 1
reason: 雖然文中沒有明確提到這一點，但關心組員的狀況有助於確保大家在同一頁，因此可以推斷出這個陳述的合理性。

statement: 提高整體合作的效率是最終目標。
verdict: 1
reason: 文中提到要分工合作才能完成整個Project，這暗示了提高合作效率的重要性，因此可以推斷出這個陳述的合理性。


Evaluating: 100%|██████████| 200/200 [01:55<00:00,  1.73it/s]


statement: 根據蔡易辰的經驗，使用MySQL的優點是因為上學期（大二上）才剛學過。
verdict: 1
reason: 蔡易辰提到使用MySQL的原因是因為上學期剛學過，因此這個陳述可以直接推斷。

statement: 對於學生來說，使用MySQL比較熟悉。
verdict: 0
reason: 雖然蔡易辰提到使用MySQL的原因，但並未明確指出學生對MySQL的熟悉程度，因此無法直接推斷。

statement: 學生可以快速上手使用MySQL。
verdict: 0
reason: 蔡易辰提到上學期剛學過MySQL，但並未提到學生能否快速上手，因此無法直接推斷。

statement: 蔡易辰強烈建議不要只用MySQL這個資料庫。
verdict: 1
reason: 蔡易辰在上下文中明確提到不建議只用MySQL，因此這個陳述可以直接推斷。

statement: 在SA階段還有更多時間學習其他技術。
verdict: 1
reason: 蔡易辰提到在SA階段有更多時間學習其他程式語言，因此這個陳述可以直接推斷。

statement: 蔡易辰並未明確提到MySQL的缺點。
verdict: 1
reason: 蔡易辰在上下文中並未提到MySQL的缺點，因此這個陳述可以直接推斷。

statement: 如果專題要用另一個語言開發，可能會需要重新學習。
verdict: 1
reason: 蔡易辰提到如果專題要用另一個語言開發，會需要花時間從頭學習，因此這個陳述可以直接推斷。

statement: 這意味著若選擇MySQL而未掌握其他資料庫，轉換時可能會影響專題的進度。
verdict: 0
reason: 雖然蔡易辰提到需要學習其他技術，但並未明確提到選擇MySQL會影響專題進度，因此無法直接推斷。

statement: 目前沒有彭文暘的相關經驗資料。
verdict: 1
reason: 上下文中並未提到彭文暘的經驗，因此這個陳述可以直接推斷。

statement: 無法提供彭文暘的觀點。
verdict: 1
reason: 由於上下文中沒有提到彭文暘的觀點，因此這個陱述可以直接推斷。
{'faithfulness': 0.8351, 'context_precision': 0.9500, 'context_recall': 0.9

In [30]:
from ragas import evaluate
from ragas.metrics import (
    AnswerCorrectness,
    ContextPrecision,
    ContextRecall,
    Faithfulness,
    AnswerRelevancy,
)
import openai
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.embeddings import GoogleEmbeddings
from google import genai
from dotenv import load_dotenv
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import OpenAIEmbeddings

load_dotenv()

import pandas as pd
import ast
from ragas import EvaluationDataset

# 讀取 CSV
df = pd.read_csv("ragas_eval_with_response_2026-05-22.csv", encoding="utf-8-sig")

# 還原 list 欄位
df["retrieved_contexts"] = df["retrieved_contexts"].apply(ast.literal_eval)

# 欄位重新命名
ragas_df = df[["user_input", "retrieved_contexts", "response", "reference"]]

rag_results = EvaluationDataset.from_pandas(ragas_df)


# 用 AsyncOpenAI
async_client = openai.AsyncOpenAI()

evaluator_llm = llm_factory("gpt-4o-mini", provider="openai", client=async_client,max_tokens=4096)
ragas_embeddings = LangchainEmbeddingsWrapper(
    OpenAIEmbeddings(model="text-embedding-3-small")
)

answer_relevancy = AnswerRelevancy(llm=evaluator_llm, embeddings=ragas_embeddings)
answer_relevancy.question_generation.instruction = """根據給定的答案，反推出最有可能對應的問題，並判斷該答案是否為模糊回答。
如果答案是模糊的，noncommittal 給 1；如果答案是明確的，noncommittal 給 0。
模糊回答是指那些迴避、含糊或不明確的回答。
例如，「我不知道」或「我不確定」都屬於模糊回答。
請用繁體中文產生問題。"""

scores = evaluate(
    dataset=rag_results,
    metrics=[
        Faithfulness(llm=evaluator_llm),
        ContextPrecision(llm=evaluator_llm),
        ContextRecall(llm=evaluator_llm),
        answer_relevancy
        # AnswerCorrectness(llm=evaluator_llm),
    ],
)


print(scores)

C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerCorrectness
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_11952\2593691224.py:2: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import (
C:\Users\User\AppData\Local\Temp\ipykernel_1195

{'faithfulness': 0.7917, 'context_precision': 0.9615, 'context_recall': 0.9038, 'answer_relevancy': 0.9091}


ragas_eval_with_response_v3
ragas_testset_v2
{'context_precision': 0.9833, 'context_recall': 0.9000, 'answer_relevancy': 0.6260, 'answer_correctness': 0.7897}

3.1
{'context_precision': 0.9833, 'context_recall': 0.9000, 'answer_relevancy': 0.6260, 'answer_correctness': 0.7897}

"ragas_eval_with_response_2026-05-22.csv"

{
    "faithfulness": 0.7917,
    "context_precision": 0.9615,
    "context_recall": 0.9038,
    "answer_relevancy": 0.9091,
}

ragas_eval_with_response_v3
{'faithfulness': 0.8351, 'context_precision': 0.9500, 'context_recall': 0.9333, 'answer_relevancy': 0.7727}